# From Tells to Cyprus
## Foundation models for archaeological survey prioritisation
**Participant challenge · four hours · Python / Google Colab · no paid services required**

You support an archaeological team with time to review **50 image tiles**. Can a
frozen foundation model improve their shortlist with only **up to 100 training
examples per class**? What happens when its image features are transferred to Cyprus?

**You will produce:** a ranked CSV, a validation comparison, a label-budget experiment,
a Cyprus transfer review, and a short recommendation to an archaeologist.

This notebook is self-contained. Upload it at [Google Colab](https://colab.research.google.com/)
using **File → Upload notebook**. A standard CPU runtime is enough for the default path:
verified DINOv2 features and 12 real Cyprus Sentinel-2 chips are embedded below.
The first data cell downloads a **600-image, balanced Enki sample (~73 MB transfer)** using
indexed HTTP byte ranges. It does not download the complete 1.7 GB training or 12 GB test
archive. Allow roughly 400 MB of free disk for the workshop.
Optional feature extraction uses a small pretrained model and benefits from a T4 GPU;
GPU availability in Colab is not guaranteed.

**Run cells in order.** Complete the four TODO cells and their checks as you go. Do not use Run all until those exercises are complete.

### Schedule and scientific framing

| Time | Task |
|---|---|
| 00:00–00:20 | Briefing; launch data download |
| 00:20–00:45 | Inspect imagery, labels and duplicate audit |
| 00:45–01:10 | Implement ranking metrics; run baseline |
| 01:10–01:55 | Frozen foundation-model features and model selection |
| 01:55–02:05 | Break |
| 02:05–02:40 | Label efficiency and error analysis |
| 02:40–03:15 | Cyprus catalogue and image-transfer audit |
| 03:15–03:35 | Freeze model; export shortlist and recommendation |
| 03:35–04:00 | Presentations and debrief |

Enki's labels concern tells and mound-like features in rendered RGB imagery. A high score
is a review priority, not evidence of a newly discovered site. The Cyprus catalogue records
settlement locations and occupation periods; it does **not** provide image visibility labels.
Our Cyprus images are **1 km windows at 10 m/pixel** from a single Sentinel-2 scene.
This introduces resolution, rendering, landscape and target differences simultaneously.
The Cyprus task is a stress test of interpretation, not a numerical detection benchmark.

### 1. Setup
Install only the small core dependencies. Existing Colab PyTorch is not replaced.

In [ ]:
import importlib.util, os, shutil, subprocess, sys

def install_packages(*requirements):
    if importlib.util.find_spec("pip") is not None:
        command = [sys.executable, "-m", "pip", "install", "-q"]
    elif shutil.which("uv"):
        command = [shutil.which("uv"), "pip", "install", "--python", sys.executable, "-q"]
    else:
        raise RuntimeError("Install pip or uv for this Python environment, then rerun this cell.")
    subprocess.check_call(command + list(requirements))

if not os.environ.get("ARCHAEOLOGY_SKIP_INSTALL"):
    install_packages("imagehash==4.3.2", "ipywidgets>=8,<9", "pillow>=10,<14", "scikit-learn>=1.4,<2",
                     "pandas>=2.1,<4", "matplotlib>=3.8,<4", "requests>=2.31,<3", "tqdm>=4.66,<5")
from IPython.display import display
print("Core dependencies ready.")


In [ ]:
"""Shared source embedded into both standalone Colab notebooks by build_notebooks.py."""
from pathlib import Path
import base64
import gzip
import hashlib
import io
import json
import os
import random
import time
import zipfile
import struct
import zlib

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, PrecisionRecallDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ENKI_URL = "https://zenodo.org/records/14950565/files/Val_norm.zip?download=1"
ENKI_MD5 = "fbd56ed8039b946eadb6481badf80c95"
ENKI_TRAIN_URL = "https://zenodo.org/records/14950565/files/Normalized_Tell_combined_350.zip?download=1"
CYPRUS_REVISION = "70a5c3d399a2451e1d8dd5eb947eb80055276b44"
CYPRUS_URL = f"https://raw.githubusercontent.com/K-A-Crawford/Cyprus_Settlements/{CYPRUS_REVISION}/Cyprus_Settlements.csv"
MODEL_ID = "facebook/dinov2-small"
MODEL_REVISION = "ed25f3a31f01632728cabb09d1542f84ab7b0056"
PREPROCESS_ID = "rgb-full-frame-bicubic-224-imagenet-mean-std-cls-l2-v1"
CYPRUS_ITEM = "S2A_MSIL2A_20240805T082601_R021_T36SVD_20240805T144559"
ROOT = Path(os.environ.get("ARCHAEOLOGY_WORKSHOP_ROOT", "archaeology_workshop"))
ROOT.mkdir(parents=True, exist_ok=True)


def file_digest(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def download(url, destination, md5=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and (md5 is None or file_digest(destination, "md5") == md5):
        return destination
    part = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(30, 120)) as response:
                response.raise_for_status()
                total = int(response.headers.get("content-length", 0)) or None
                with open(part, "wb") as out, tqdm(total=total, unit="B", unit_scale=True,
                                                    desc=destination.name) as progress:
                    for block in response.iter_content(1024 * 1024):
                        out.write(block)
                        progress.update(len(block))
            if md5 is not None and file_digest(part, "md5") != md5:
                raise ValueError("Checksum mismatch: the download is incomplete or has changed.")
            part.replace(destination)
            return destination
        except (requests.RequestException, ValueError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)


def download_enki_sample(manifest_path, destination):
    """Fetch indexed byte ranges, validate ZIP-member CRCs, build a small local ZIP.

    The embedded manifest was prepared from the original ZIP central directory. If a server
    ignores byte ranges, abort rather than accidentally downloading the 1.7 GB archive.
    """
    manifest = json.loads(Path(manifest_path).read_text())
    rows = manifest["members"]
    expected = {r["member"]: r for r in rows}
    destination = Path(destination)
    if destination.exists():
        with zipfile.ZipFile(destination) as z:
            if set(z.namelist()) == set(expected) and all(
                z.getinfo(name).CRC == row["crc"] for name, row in expected.items()
            ) and z.testzip() is None:
                return destination
    cache_dir = destination.parent / "range_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(".part")
    groups = sorted(set(r["batch"] for r in rows))
    with requests.Session() as session, zipfile.ZipFile(partial, "w", zipfile.ZIP_DEFLATED) as output:
        for batch in tqdm(groups, desc="Download compact balanced Enki sample"):
            group = [r for r in rows if r["batch"] == batch]
            start = min(r["offset"] for r in group)
            end = max(r["span_end"] for r in group)
            cached = cache_dir / f"{start}-{end}.bin"
            if cached.exists() and cached.stat().st_size == end - start + 1:
                data = cached.read_bytes()
            else:
                for attempt in range(4):
                    try:
                        with session.get(ENKI_TRAIN_URL, headers={"Range": f"bytes={start}-{end}"},
                                         stream=True, timeout=(30, 120)) as response:
                            if response.status_code == 429:
                                delay = min(60, int(response.headers.get("Retry-After", "30")))
                                time.sleep(delay)
                                continue
                            response.raise_for_status()
                            if response.status_code != 206:
                                raise RuntimeError("Server did not honour byte ranges. Use the facilitator's enki_workshop_sample.zip instead.")
                            actual_range = response.headers.get("Content-Range", "")
                            if not actual_range.startswith(f"bytes {start}-{end}/"):
                                raise RuntimeError(f"Unexpected byte range: {actual_range}")
                            data = response.content
                        if len(data) != end - start + 1:
                            raise ValueError("Truncated range download")
                        cached.write_bytes(data)
                        break
                    except requests.RequestException:
                        if attempt == 3:
                            raise
                        time.sleep(2 ** attempt)
                else:
                    raise RuntimeError("Zenodo rate limit persisted; retry later or use the facilitator's sample ZIP.")
            for row in group:
                position = row["offset"] - start
                assert data[position:position + 4] == b"PK\x03\x04"
                name_size, extra_size = struct.unpack_from("<HH", data, position + 26)
                payload_start = position + 30 + name_size + extra_size
                compressed = data[payload_start:payload_start + row["compressed_size"]]
                if row["method"] == 8:
                    payload = zlib.decompress(compressed, -15)
                elif row["method"] == 0:
                    payload = compressed
                else:
                    raise ValueError("Unsupported ZIP compression method")
                assert len(payload) == row["file_size"]
                assert zlib.crc32(payload) & 0xFFFFFFFF == row["crc"], "Archive member checksum mismatch"
                info = zipfile.ZipInfo(row["member"], date_time=(2025, 3, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                output.writestr(info, payload)
    partial.replace(destination)
    return destination


def index_enki(archive, image_dir):
    """Read only image members, infer classes from actual path components, anonymise paths."""
    image_dir = Path(image_dir)
    image_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    with zipfile.ZipFile(archive) as source:
        members = sorted(source.namelist())
        for name in tqdm(members, desc="Index and audit images"):
            parts = Path(name).parts
            if "__MACOSX" in parts or any(p.startswith(".") for p in parts):
                continue
            if Path(name).suffix.lower() not in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}:
                continue
            classes = set(p.lower() for p in parts) & {"positive", "tell", "negative"}
            if len(classes) != 1:
                raise ValueError(f"Unrecognised class directory: {name}")
            label = int(bool(classes & {"positive", "tell"}))
            tile_id = hashlib.sha256(name.encode()).hexdigest()[:16]
            target = image_dir / f"{tile_id}.png"
            with Image.open(io.BytesIO(source.read(name))) as original:
                rgb = original.convert("RGB")
                pixel_hash = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
                phash = str(imagehash.phash(rgb))
                if not target.exists():
                    rgb.save(target)
                width, height = rgb.size
            rows.append(dict(tile_id=tile_id, member=name, path=str(target), label=label,
                             category=parts[-2], pixel_sha256=pixel_hash, phash=phash,
                             width=width, height=height))
    frame = pd.DataFrame(rows).sort_values("tile_id").reset_index(drop=True)
    assert frame.tile_id.is_unique and set(frame.label) == {0, 1}
    return frame


def deduplicate_images(frame, max_hamming=4):
    """Connected components of close pHashes; exclude mixed-label components; keep one/image family.

    This is an image-similarity audit, NOT a verified geographic/site grouping.
    """
    hashes = [int(x, 16) for x in frame.phash]
    parent = list(range(len(frame)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, value in enumerate(hashes):
        for j in range(i):
            if (value ^ hashes[j]).bit_count() <= max_hamming:
                parent[find(i)] = find(j)
    audit = frame.copy()
    audit["image_family"] = [f"family_{find(i):04d}" for i in range(len(frame))]
    conflicts = audit.groupby("image_family").label.nunique()
    bad = set(conflicts[conflicts > 1].index)
    clean = (audit.loc[~audit.image_family.isin(bad)]
             .drop_duplicates("image_family").reset_index(drop=True))
    report = {
        "raw_images": len(frame), "exact_duplicate_images": int(frame.pixel_sha256.duplicated().sum()),
        "similarity_families": audit.image_family.nunique(), "mixed_label_families_excluded": len(bad),
        "retained_images": len(clean), "phash_max_hamming": max_hamming,
        "geographic_independence_verified": False,
    }
    return clean, audit, report


def workshop_split(clean, per_class=100):
    # One representative per similarity family has already been selected.
    trainval, test = train_test_split(clean, test_size=0.2, stratify=clean.label, random_state=SEED)
    train_pool, val = train_test_split(trainval, test_size=0.25,
                                     stratify=trainval.label, random_state=SEED)
    train = pd.concat([g.sample(min(per_class, len(g)), random_state=SEED)
                       for _, g in train_pool.groupby("label")])
    splits = {"train": train, "validation": val, "test": test}
    for name, part in splits.items():
        splits[name] = part.sort_values("tile_id").reset_index(drop=True)
    assert not (set(train.image_family) & set(val.image_family))
    assert not (set(train.image_family) & set(test.image_family))
    assert not (set(val.image_family) & set(test.image_family))
    return splits


def contact_sheet(frame, scores=None, n=12, title="Image tiles", show_labels=True):
    subset = frame.head(n).reset_index(drop=True)
    if subset.empty:
        print("No images in this selection.")
        return
    fig, axes = plt.subplots(int(np.ceil(len(subset) / 4)), 4,
                             figsize=(12, 3.1 * int(np.ceil(len(subset) / 4))), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i, row in subset.iterrows():
        ax = axes.flat[i]
        with Image.open(row.path) as im:
            ax.imshow(im)
        label = f" | y={row.label}" if show_labels and "label" in row else ""
        score = f" | score={float(scores[i]):.3f}" if scores is not None else ""
        ax.set_title(f"{row.tile_id[:9]}{label}{score}", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def handcrafted_features(paths):
    """Colour distributions, global moments and simple gradient histograms (no pretrained network)."""
    features = []
    for path in tqdm(paths, desc="Colour and texture features"):
        with Image.open(path) as im:
            rgb = np.asarray(im.convert("RGB").resize((64, 64)), dtype=np.float32) / 255
        colour = np.concatenate([np.histogram(rgb[:, :, c], bins=16, range=(0, 1), density=False)[0]
                                 / (64 * 64) for c in range(3)])
        grey = rgb.mean(axis=2)
        dy, dx = np.gradient(grey)
        magnitude = np.sqrt(dx ** 2 + dy ** 2)
        angle = np.arctan2(dy, dx)
        orientation = np.histogram(angle, bins=12, range=(-np.pi, np.pi), weights=magnitude)[0]
        orientation /= orientation.sum() + 1e-8
        moments = np.r_[rgb.mean(axis=(0, 1)), rgb.std(axis=(0, 1)), magnitude.mean(), magnitude.std()]
        features.append(np.r_[colour, moments, orientation])
    return np.asarray(features, dtype=np.float32)


def extract_dino(paths, local_model=None, batch_size=16):
    """Frozen CLS embeddings, full-frame resize to 224x224, ImageNet normalisation, L2 normalisation.

    Model weights and preprocessing are pinned. The same function is used for Enki and Cyprus.
    Imports are lazy so the bundled embedding path also works without torch/transformers.
    """
    import torch
    from transformers import AutoModel

    device = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu")
    torch.manual_seed(SEED)
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    if local_model:
        model = AutoModel.from_pretrained(str(local_model), local_files_only=True)
    else:
        model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
                                          cache_dir=str(ROOT / "model_cache"))
    model = model.eval().to(device)
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    vectors = []
    started = time.perf_counter()
    with torch.inference_mode():
        for start in tqdm(range(0, len(paths), batch_size), desc=f"DINOv2 ({device})"):
            batch = []
            for path in paths[start:start + batch_size]:
                with Image.open(path) as im:
                    arr = np.asarray(im.convert("RGB").resize((224, 224), Image.Resampling.BICUBIC),
                                     dtype=np.float32).copy() / 255
                batch.append((torch.from_numpy(arr).permute(2, 0, 1) - mean) / std)
            tensor = torch.stack(batch).to(device)
            feature = model(pixel_values=tensor).last_hidden_state[:, 0]
            feature = torch.nn.functional.normalize(feature.float(), dim=1)
            vectors.append(feature.cpu().numpy())
    print(f"Encoded {len(paths)} images in {time.perf_counter() - started:.1f}s on {device}.")
    return np.concatenate(vectors).astype(np.float32)


def load_embedding_cache(frame, cache_path):
    with np.load(cache_path, allow_pickle=False) as cache:
        assert str(cache["model_id"]) == MODEL_ID
        assert str(cache["revision"]) == MODEL_REVISION
        assert str(cache["preprocessing"]) == PREPROCESS_ID
        positions = {str(k): i for i, k in enumerate(cache["tile_ids"])}
        indices = [positions[k] for k in frame.tile_id]
        expected = cache["pixel_sha256"][indices].astype(str)
        assert np.array_equal(expected, frame.pixel_sha256.to_numpy()), "Image/cache mismatch."
        return cache["embeddings"][indices].astype(np.float32)


def partition_features(frame, features, splits):
    positions = {key: i for i, key in enumerate(frame.tile_id)}
    return {name: features[[positions[k] for k in subset.tile_id]]
            for name, subset in splits.items()}


def ranking_indices(scores):
    values = np.asarray(scores, dtype=float)
    if values.ndim != 1 or not np.isfinite(values).all():
        raise ValueError("Scores must be a finite one-dimensional array.")
    # Stable tie-breaking follows the tile_id-sorted input manifest.
    return np.argsort(-values, kind="stable")


def nearest_examples(query_vector, train_vectors, train_frame, k=4):
    similarity = train_vectors @ query_vector
    order = ranking_indices(similarity)[:k]
    return train_frame.iloc[order].reset_index(drop=True), similarity[order]


def learning_curve_rows(estimator, X, y, X_val, y_val, metric_fn, k=50):
    rows = []
    for budget in (10, 25, 50, 100):
        for seed in (0, 1, 2):
            rng = np.random.default_rng(seed)
            if any((y == label).sum() < budget for label in (0, 1)):
                continue
            indices = np.concatenate([rng.choice(np.flatnonzero(y == label), budget, replace=False)
                                      for label in (0, 1)])
            fitted = clone(estimator).fit(X[indices], y[indices])
            scores = fitted.predict_proba(X_val)[:, 1]
            rows.append(dict(labels_per_class=budget, seed=seed, **metric_fn(y_val, scores, k)))
    return pd.DataFrame(rows)


def write_submission(frame, scores, destination):
    scores = np.asarray(scores, dtype=float)
    if len(scores) != len(frame):
        raise ValueError("Exactly one score per test tile is required.")
    order = ranking_indices(scores)
    output = pd.DataFrame({"tile_id": frame.tile_id.to_numpy()[order], "score": scores[order],
                           "rank": np.arange(1, len(scores) + 1)})
    assert output.tile_id.is_unique
    output.to_csv(destination, index=False)
    return output


def score_submission(submission, truth, metric_fn, k=50):
    if not submission.tile_id.is_unique or set(submission.tile_id) != set(truth.tile_id):
        raise ValueError("Submission must contain every test tile exactly once, with no extra IDs.")
    aligned = truth[["tile_id", "label"]].merge(submission[["tile_id", "score"]],
                                               on="tile_id", how="left", validate="one_to_one")
    return metric_fn(aligned.label.to_numpy(), aligned.score.to_numpy(), k)


def make_cyprus_bundle(phases, output_dir, local_model=None):
    """Instructor refresh utility: public Sentinel-2 COG windows; no paid account required.

    Normal participant runs use the verified, embedded bundle instead of live STAC requests.
    """
    import rasterio
    from rasterio.warp import transform as transform_points
    from rasterio.windows import Window
    import planetary_computer

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    item_url = f"https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a/items/{CYPRUS_ITEM}"
    response = requests.get(item_url, timeout=60)
    response.raise_for_status()
    item = response.json()
    points = phases.drop_duplicates("ID").copy()
    for c in ("Long_Cord", "Lat_Cord", "Longitude", "Latitude"):
        points[c] = pd.to_numeric(points[c], errors="coerce")
    candidates = points.loc[(points.LocQual.str.strip() == "A") &
                            points.Long_Cord.between(32.4, 32.85) &
                            points.Lat_Cord.between(34.72, 35.12)]
    candidates = candidates.sort_values("ID").sample(frac=1, random_state=SEED)
    rows = []
    known_xy = points[["Longitude", "Latitude"]].dropna().to_numpy()
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="3",
                       GDAL_HTTP_RETRY_DELAY="2"):
        with rasterio.open(planetary_computer.sign(item["assets"]["visual"]["href"])) as rgb, \
             rasterio.open(planetary_computer.sign(item["assets"]["SCL"]["href"])) as scl:
            assert rgb.crs.to_epsg() == 32636
            for _, site in candidates.iterrows():
                if len(rows) >= 12:
                    break
                x, y = transform_points("EPSG:4326", rgb.crs, [site.Long_Cord], [site.Lat_Cord])
                pair = []
                for kind, px, py in (("catalogued_location", x[0], y[0]),
                                     ("unlabelled_background", x[0] + 2000, y[0])):
                    if kind == "unlabelled_background" and np.linalg.norm(known_xy - [px, py], axis=1).min() < 750:
                        break
                    rr, cc = rgb.index(px, py)
                    if not (50 <= rr < rgb.height - 50 and 50 <= cc < rgb.width - 50):
                        break
                    # 1 km x 1 km at 10 m/pixel; SCL has 20 m pixels.
                    image = rgb.read([1, 2, 3], window=Window(cc - 50, rr - 50, 100, 100))
                    sr, sc = scl.index(px, py)
                    quality = scl.read(1, window=Window(sc - 25, sr - 25, 50, 50))
                    clear_land = float(np.isin(quality, [4, 5, 7]).mean())
                    if clear_land < 0.9 or (image.sum(axis=0) == 0).mean() > 0.05:
                        break
                    tile_id = f"{site.ID}_{'site' if kind == 'catalogued_location' else 'background'}"
                    filename = tile_id + ".png"
                    Image.fromarray(image.transpose(1, 2, 0)).save(output_dir / filename)
                    lon, lat = transform_points(rgb.crs, "EPSG:4326", [px], [py])
                    pair.append(dict(tile_id=tile_id, filename=filename, kind=kind, paired_site=site.ID,
                                     lon=lon[0], lat=lat[0], LocQual=site.LocQual.strip(),
                                     clear_land_fraction=clear_land, gsd_m=10, footprint_m=1000,
                                     acquisition=item["properties"]["datetime"], item_id=CYPRUS_ITEM))
                if len(pair) == 2:
                    rows.extend(pair)
    manifest = pd.DataFrame(rows).sort_values("tile_id").reset_index(drop=True)
    if len(manifest) != 12:
        raise RuntimeError(f"Expected six usable pairs; found {len(manifest)} chips.")
    manifest.to_csv(output_dir / "manifest.csv", index=False)
    vectors = extract_dino([str(output_dir / f) for f in manifest.filename], local_model=local_model)
    np.savez_compressed(output_dir / "embeddings.npz", tile_ids=manifest.tile_id.to_numpy(dtype=str),
                        embeddings=vectors, model_id=MODEL_ID, revision=MODEL_REVISION,
                        preprocessing=PREPROCESS_ID)
    metadata = dict(item_url=item_url, item_id=CYPRUS_ITEM, acquisition=item["properties"]["datetime"],
                    attribution="Contains modified Copernicus Sentinel data (2024), accessed through Microsoft Planetary Computer.",
                    interpretation="Landscape-context stress test, not archaeological detection ground truth.")
    (output_dir / "provenance.json").write_text(json.dumps(metadata, indent=2))
    return manifest


The next collapsed cell unpacks the embedded **frozen features**, Cyprus catalogue and
Cyprus sample. Features are bound to image pixel hashes, model revision and preprocessing.
They contain no fitted classifier or solution to the exercises. The full Enki imagery is
downloaded from its original repository in the next section.

In [ ]:
# Embedded reproducible workshop assets; no edits needed.
ASSET_B64 = "..."
assets = ROOT / "assets"
assets.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(ASSET_B64))) as bundle:
    for member in bundle.infolist():
        target = (assets / member.filename).resolve()
        if not target.is_relative_to(assets.resolve()):
            raise ValueError("Unsafe bundle path")
    bundle.extractall(assets)
del ASSET_B64
print("Unpacked frozen embeddings, Cyprus catalogue and 12 Sentinel-2 chips.")


### 2. Load and audit Enki
We use **600 images from the published training archive**: 300 negatives and 300 positives
sampled across five RGB landscape-context folders. The explicitly masked and black-and-white
positive folders are excluded. The fixed manifest samples blocks of ten files to reduce network
requests; this is a curated teaching sample, not a representative prevalence estimate.
Our train/validation/test names are workshop splits, not the authors' original experiment.
Labels come from the archive's `Tell` and `Negative` directories; macOS sidecars are excluded.
Context subfolders are retained for error analysis, never as features. Byte ranges and ZIP-member
CRCs are verified, then decoded image hashes are checked against the embedding cache.

We group images with close 64-bit perceptual hashes (Hamming distance ≤4), exclude groups
with contradictory labels, and retain one representative per group. This can miss shifted,
rotated or differently rendered views of the same site. Similarity is not a site identifier.

In [ ]:
archive = download_enki_sample(assets / "sample_manifest.json", ROOT / "enki_workshop_sample.zip")
all_images = index_enki(archive, ROOT / "images")
clean_images, duplicate_audit, audit_report = deduplicate_images(all_images)
display(pd.Series(audit_report, name="audit"))
display(all_images.groupby(["label", "category"]).size().rename("images").to_frame())
print("Image dimensions:", all_images[["width", "height"]].value_counts().to_dict())


In [ ]:
splits = workshop_split(clean_images, per_class=100)
train, validation, test = (splits[k] for k in ("train", "validation", "test"))
REVIEW_BUDGET = 50
display(pd.DataFrame({name: part.label.value_counts().sort_index()
                      for name, part in splits.items()}).rename_axis("label"))
contact_sheet(train.sample(12, random_state=SEED), title="Training examples: 1=tell; 0=background")
# The remaining unused training-pool images are not extra labels for this challenge.
print("Do not inspect test labels or test images until model selection is frozen.")


#### What is in this teaching sample?
The left chart counts the **original sampled images** by archive context and label; these
are source categories, not independent regions. The right chart shows what remains after
similarity filtering and how the retained images are allocated. Unused images belong to
the training pool but are withheld to enforce the label budget. The balanced sampling
was designed for teaching and does not estimate the prevalence of tells in the landscape.

In [ ]:
counts = (all_images.groupby(["category", "label"]).size().unstack(fill_value=0)
          .reindex(columns=[0, 1], fill_value=0).sort_index())
used = [len(train), len(validation), len(test)]
allocation = used + [len(clean_images) - sum(used)]
fig, (ax_context, ax_split) = plt.subplots(1, 2, figsize=(13, 5),
                                           gridspec_kw={"width_ratios": [1.35, 1]})
positions = np.arange(len(counts))
ax_context.barh(positions, counts[0], color="#4477aa", label="Source negative (0)")
ax_context.barh(positions, counts[1], left=counts[0], color="#ee7733", label="Source positive (1)")
ax_context.set_yticks(positions, labels=counts.index.astype(str))
ax_context.invert_yaxis()
ax_context.set(xlabel="Images before filtering", title="Archive context in the 600-image sample")
ax_context.legend(loc="lower right", fontsize=8)
bars = ax_split.barh(["Train labels", "Validation", "Test", "Unused train pool"], allocation,
                     color=["#4477aa", "#66c2a5", "#aa4499", "#bbbbbb"])
ax_split.invert_yaxis()
ax_split.bar_label(bars, padding=3)
ax_split.set(xlabel="Retained images", title=f"Allocation after filtering (n={len(clean_images)})")
ax_split.margins(x=0.16)
fig.tight_layout()
plt.show()


**Discuss:** What visual evidence distinguishes the classes? Could rendering, masking or
land-cover context be a shortcut? Why does duplicate removal not establish geographic transfer?
Raw accuracy is a poor description of a ranking system with a limited review budget.

**Why keep three partitions?** Training fits candidate models; validation compares them;
the untouched test set provides one final assessment after the choice is frozen. Similarity
filtering reduces image duplication but does not make these independent geographic regions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
roles = [
    ("TRAIN", "Fit candidate models", "#e1eff0"),
    ("VALIDATION", "Compare and select", "#f8edda"),
    ("TEST", "Assess the frozen choice", "#efe8f1"),
]
for i, (name, role, colour) in enumerate(roles):
    x = .04 + i * .34
    ax.add_patch(plt.Rectangle((x, .25), .26, .5, transform=ax.transAxes,
                               facecolor=colour, edgecolor="none"))
    ax.text(x + .13, .58, name, transform=ax.transAxes, ha="center", weight="bold")
    ax.text(x + .13, .40, role, transform=ax.transAxes, ha="center", fontsize=10)
    if i < 2:
        ax.annotate("", xy=(x + .33, .5), xytext=(x + .27, .5),
                    xycoords="axes fraction", arrowprops={"arrowstyle": "->", "lw": 1.5})
ax.set_title("Fit → select → freeze → assess")
ax.axis("off")
fig.tight_layout()
plt.show()


### 3. Exercise 1 — evaluate the archaeologist's shortlist
Implement `ranking_metrics(y_true, scores, k)` returning:
- `AP`: average precision over the full ranked set.
- `recall_at_k`: positives in the top k / all positives.
- `precision_at_k`: positives in the top k / number actually reviewed.
- `hits` and `reviewed`: integer counts.

Use `min(k, number_of_tiles)` as the effective review budget. With no positives,
AP and recall are undefined: return `np.nan`. Rank by score descending; score ties follow
the tile-ID order. A score is not automatically a calibrated archaeological probability.

In [ ]:
# A schematic, not a result from the Enki test set.
from matplotlib.patches import Patch
example_labels = [1, 0, 1, 0, 0, 1, 0, 1, 0, 0]
example_budget = 4
fig, ax = plt.subplots(figsize=(11, 2.8))
colors = ["#ee7733" if label else "#4477aa" for label in example_labels]
ax.bar(np.arange(1, 11), np.ones(10), color=colors, edgecolor="white", width=0.85)
ax.axvline(example_budget + 0.5, color="#222222", linestyle="--", linewidth=2)
ax.text(2.5, 1.08, "Reviewed first", ha="center", fontsize=10)
ax.text(7.5, 1.08, "Not reviewed", ha="center", fontsize=10)
ax.set_xticks(np.arange(1, 11))
ax.set(xlabel="Rank (highest model score on the left)", yticks=[],
       title="A fixed review budget turns scores into a shortlist")
ax.legend(handles=[Patch(color="#ee7733", label="Source positive"),
                   Patch(color="#4477aa", label="Source negative")],
          loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
ax.set_ylim(0, 1.28)
fig.tight_layout()
plt.show()


In this illustration, each rectangle is a tile and the dashed line is the review limit.
Precision asks what fraction of reviewed tiles are positive; recall asks what fraction of
all positives were reviewed. The colors represent known labels for evaluation, not information
available to a reviewer before labels are known. Average precision considers the full ranking.

In [ ]:
def ranking_metrics(y_true, scores, k=50):
    # TODO 1: implement the metric dictionary described above.
    # ranking_indices(scores) provides descending, stable ordering.
    # Use average_precision_score for AP. Clip k to the number of tiles.
    # Return NaN for AP/recall if there are no positive labels.
    raise NotImplementedError("Complete Exercise 1: ranking under a review budget")


In [ ]:
toy = ranking_metrics(np.array([1, 0, 1, 0]), np.array([.9, .8, .7, .1]), k=2)
assert np.isclose(toy["recall_at_k"], .5)
assert np.isclose(toy["precision_at_k"], .5)
assert np.isclose(toy["AP"], 5 / 6)
assert toy["hits"] == 1 and toy["reviewed"] == 2
assert ranking_metrics([1, 0], [.2, .1], k=10)["reviewed"] == 2
assert np.isnan(ranking_metrics([0, 0], [.9, .1], k=1)["recall_at_k"])
print("Metric checks passed.")


### 4. Conventional features and foundation-model features
The conventional baseline uses RGB histograms, moments and gradient statistics. DINOv2-small
supplies 384-dimensional frozen CLS-token embeddings. It is a general visual foundation
model, **not an EO-specialised model**. Comparing its features to the baseline is the main
experiment; an EO-specific encoder such as RemoteCLIP is a follow-on experiment.

Both representations use the same tiles and training labels. The bundled DINOv2 features
were computed without adapting the encoder to any workshop labels. Each full image is
resized to 224×224, normalised using ImageNet channel statistics, and the CLS vector is L2
normalised. We use the same preprocessing for Cyprus.

In [ ]:
X_handcrafted = handcrafted_features(clean_images.path.tolist())
X_dino = load_embedding_cache(clean_images, assets / "enki_embeddings.npz")
feature_sets = {
    "colour_texture": partition_features(clean_images, X_handcrafted, splits),
    "dinov2": partition_features(clean_images, X_dino, splits),
}
y_train = train.label.to_numpy()
y_val = validation.label.to_numpy()
print("Feature shapes:", X_handcrafted.shape, X_dino.shape)
print("Pinned model:", MODEL_ID, MODEL_REVISION)


**Optional GPU exercise:** set `RECOMPUTE_DINO=True` below. In Colab, select
**Runtime → Change runtime type → T4 GPU** before starting if one is available.
The default cached path needs no model download. Recomputing downloads about 90 MB of
weights and uses batches of 16 at 224×224; it does not fine-tune the model.
A CPU also works, but allow additional time. This cell must precede model fitting.

In [ ]:
RECOMPUTE_DINO = False
if RECOMPUTE_DINO:
    install_packages("transformers==4.57.6")
    fresh = extract_dino(clean_images.path.tolist())
    print("Mean cosine agreement with cache:", np.mean(np.sum(fresh * X_dino, axis=1)))
    X_dino = fresh
    feature_sets["dinov2"] = partition_features(clean_images, X_dino, splits)
else:
    print("Using verified cached embeddings; no GPU required.")


### 5. Exercise 2 — a lightweight classifier
Return a scikit-learn pipeline with `StandardScaler` followed by `LogisticRegression`.
The pipeline ensures that feature scaling is learned from training data only.
`C` controls inverse regularisation strength: smaller values mean stronger regularisation.

In [ ]:
def make_probe(C=1.0):
    # TODO 2: return a pipeline containing StandardScaler and LogisticRegression.
    # Pass C, class_weight="balanced", max_iter=2000 and random_state=SEED.
    # Do not fit anything in this function.
    raise NotImplementedError("Complete Exercise 2: the linear probe")


In [ ]:
baseline = make_probe(C=1.0).fit(feature_sets["colour_texture"]["train"], y_train)
baseline_val = baseline.predict_proba(feature_sets["colour_texture"]["validation"])[:, 1]
display(pd.Series(ranking_metrics(y_val, baseline_val, REVIEW_BUDGET), name="baseline"))
assert len(baseline.steps) == 2
assert not hasattr(make_probe().steps[0][1], "mean_")


### 6. Exercise 3 — select on validation data only
Compare the two representations with `C ∈ {0.01, 0.1, 1, 10}`. Use validation
recall at the fixed review budget as the primary criterion, then AP as a tie-breaker.
Keep the test set untouched. Even if the conventional baseline wins, report that result.

In [ ]:
# TODO 3: compare both representations and C in (0.01, 0.1, 1.0, 10.0).
# Fit on matrices["train"] / y_train; score matrices["validation"] / y_val.
# Required outputs:
# candidate_models[key] = (representation, fitted_estimator)
# validation_results: DataFrame with model, representation, C and metric columns
# best_key, best_representation, best_model
# Select highest recall_at_k, breaking ties with AP; never inspect test labels.
raise NotImplementedError("Complete Exercise 3: validation-only model selection")


In [ ]:
assert len(validation_results) == 8
assert set(validation_results.representation) == {"colour_texture", "dinov2"}
assert best_key in candidate_models
validation_results.to_csv(ROOT / "validation_results.csv", index=False)
for representation in feature_sets:
    key = validation_results.loc[validation_results.representation == representation, "model"].iloc[0]
    fitted = candidate_models[key][1]
    scores = fitted.predict_proba(feature_sets[representation]["validation"])[:, 1]
    PrecisionRecallDisplay.from_predictions(y_val, scores, name=key, ax=plt.gca())
plt.title("Validation precision–recall curves")
plt.show()


### 7. Label efficiency and failure analysis
Compare 10, 25, 50 and 100 training labels **per class**, with three random selections at each
budget. The validation set and selected regularisation stay fixed. These curves describe
variation in training-label sampling, not uncertainty across independent geographical regions.
Choose one result to explain: a small-label gain, a plateau, or a failure mode.

In [ ]:
best_dino_key = validation_results.loc[validation_results.representation == "dinov2", "model"].iloc[0]
dino_probe = candidate_models[best_dino_key][1]
curve = learning_curve_rows(dino_probe, feature_sets["dinov2"]["train"], y_train,
                           feature_sets["dinov2"]["validation"], y_val,
                           ranking_metrics, REVIEW_BUDGET)
display(curve.groupby("labels_per_class")[["AP", "recall_at_k"]].agg(["mean", "std"]))
summary = curve.groupby("labels_per_class").recall_at_k.agg(["mean", "std"])
plt.errorbar(summary.index * 2, summary["mean"], yerr=summary["std"], marker="o", capsize=4)
plt.xlabel("Total labelled training tiles")
plt.ylabel("Validation recall at review budget")
plt.ylim(0, 1)
plt.title("Frozen DINOv2: label efficiency (three training samples)")
plt.show()
curve.to_csv(ROOT / "learning_curve.csv", index=False)


In [ ]:
dino_val_scores = dino_probe.predict_proba(feature_sets["dinov2"]["validation"])[:, 1]
failures = validation.copy()
failures["score"] = dino_val_scores
false_alarms = failures.loc[failures.label == 0].sort_values("score", ascending=False)
missed = failures.loc[failures.label == 1].sort_values("score")
contact_sheet(false_alarms, false_alarms.score.to_numpy(), n=8, title="Highest-scoring labelled backgrounds")
contact_sheet(missed, missed.score.to_numpy(), n=8, title="Lowest-scoring labelled positives")
# Inspect model neighbours for one difficult validation image.
query_index = int(np.argmax(np.where(y_val == 0, dino_val_scores, -np.inf)))
neighbours, similarity = nearest_examples(feature_sets["dinov2"]["validation"][query_index],
                                          feature_sets["dinov2"]["train"], train)
contact_sheet(neighbours, similarity, n=4, title="Nearest training examples to the strongest false alarm")


**Record your interpretation:** identify two false-alarm mechanisms and one missed-positive
mechanism from the actual images. Are the nearest neighbours similar because of morphology,
colour or land use? What extra evidence would resolve the ambiguity?

In [ ]:
error_notes = {
    "false_alarm_1": "",
    "false_alarm_2": "",
    "missed_positive": "",
    "next_evidence_needed": "",
}


### 8. Exercise 4 — Cyprus catalogue audit
Load the pinned catalogue and produce one record per settlement `ID`. Occupation-phase
rows must not become independent training examples. `Long_Cord` / `Lat_Cord` contain
longitude / latitude in degrees; the misleadingly named `Longitude` / `Latitude` contain
UTM zone 36N metres. The dictionary defines location quality A–E; retain and flag other values.
Check whether all occupation rows for an ID agree on coordinates and quality. Retain IDs with
conflicting locations, flag them and leave their single-point coordinates missing; do not invent
a centroid or silently choose one location.

[Field definitions](https://github.com/K-A-Crawford/Cyprus_Settlements/blob/70a5c3d399a2451e1d8dd5eb947eb80055276b44/settlement_key.txt):
A = supplied projected coordinates; B = cadastral plans; C = maps with UTM references;
D = maps without coordinate references; E = modern locality names.
These are provenance categories, not numerical error radii.

In [ ]:
phases = pd.read_csv(assets / "Cyprus_Settlements.csv")
print("Occupation rows:", len(phases), "Unique settlement IDs:", phases.ID.nunique())
display(phases[["ID", "Period", "Longitude", "Latitude", "Long_Cord", "Lat_Cord", "LocQual"]].head())


In [ ]:
def summarise_sites(phases):
    # TODO 4: return one row per ID, with columns:
    # ID, Toponym, lon, lat, LocQual, occupation_rows, periods, quality_documented,
    # quality_conflict, coordinate_conflict.
    # lon/lat come from Long_Cord/Lat_Cord, NOT Longitude/Latitude.
    # Strip whitespace in LocQual; only A-E have documented definitions.
    # Keep unknown values and mark quality_documented=False. Join distinct quality
    # codes with "|" and flag quality_conflict if a site has multiple codes.
    # Count distinct (ID, Long_Cord, Lat_Cord) rows per ID. For sites with multiple
    # locations, set coordinate_conflict=True and lon/lat=NaN rather than choosing one.
    # Use groupby(...).agg(...) and count distinct Period values.
    raise NotImplementedError("Complete Exercise 4: occupation phases versus sites")


In [ ]:
sites = summarise_sites(phases)
assert len(sites) == phases.ID.nunique() == 1559
assert sites.ID.is_unique
assert sites.occupation_rows.sum() == len(phases) == 8766
assert sites.lon.dropna().between(31, 35).all() and sites.lat.dropna().between(34, 36).all()
assert sites.loc[sites.coordinate_conflict, ["lon", "lat"]].isna().all().all()
assert sites.loc[sites.coordinate_conflict, "ID"].tolist() == ["CY453"]
display(sites.groupby(["LocQual", "quality_documented"]).size().rename("sites").to_frame())
print("Undocumented/blank values:", sorted(sites.loc[~sites.quality_documented, "LocQual"].unique()))
display(sites.loc[sites.coordinate_conflict | sites.quality_conflict])
for quality, group in sites.groupby("LocQual"):
    plt.scatter(group.lon, group.lat, s=8, alpha=.6, label=quality or "blank")
plt.xlabel("Longitude (degrees)")
plt.ylabel("Latitude (degrees)")
plt.title("Cyprus catalogue: one point per settlement")
plt.legend(title="Location quality", fontsize=8)
plt.gca().set_aspect(1 / np.cos(np.deg2rad(35)))
plt.show()


### 9. Transfer stress test using real Cyprus imagery
The embedded sample contains six catalogue locations with `LocQual=A`, plus six nearby
**unlabelled** background locations. Background centres were placed 2 km east and checked
to be at least 750 m from catalogue points. This does not establish archaeological absence.
Windows with less than 90% clear-land pixels in Sentinel-2's scene classification were rejected.

All chips use the same pinned scene from 5 August 2024, in southwest Cyprus. They were
read from the public Planetary Computer Sentinel-2 L2A collection using COG windows.
Each chip is 100×100 pixels (1 km square) before model resizing. Display interpolation or
resizing cannot restore spatial detail missing at 10 m. Six pairs cannot establish transfer accuracy.

We deliberately transfer the Enki-trained **DINOv2 probe**, even if a different model won the
source comparison, to examine what that representation does in a different domain.

**What changes during transfer?** The model is carried across domains unchanged, but the
imagery and meaning of available labels differ. A catalogue location is not an image-visibility
label, and a nearby background location is not a verified absence. The diagram shows why these
twelve chips support an interpretive review rather than an accuracy estimate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
cards = [
    ("Source: Enki", "Rendered RGB tiles\nSource tell / negative labels\nCurated teaching split", "#e1eff0"),
    ("Same model", "Frozen DINOv2 features\nEnki-trained linear probe\nScores are review priorities", "#f8edda"),
    ("Target: Cyprus", "Sentinel-2: 10 m pixels\nCatalogue and unlabelled locations\nNo visibility ground truth", "#efe8f1"),
]
for ax, (title, details, colour) in zip(axes, cards):
    ax.set_facecolor(colour)
    ax.text(.5, .78, title, ha="center", va="center", weight="bold", fontsize=12,
            transform=ax.transAxes)
    ax.text(.5, .43, details, ha="center", va="center", fontsize=10,
            linespacing=1.8, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.suptitle("Transfer changes the evidence, not just the location", fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
cyprus_dir = assets / "cyprus"
cyprus = pd.read_csv(cyprus_dir / "manifest.csv")
cyprus["path"] = [str(cyprus_dir / f) for f in cyprus.filename]
with np.load(cyprus_dir / "embeddings.npz", allow_pickle=False) as cache:
    assert np.array_equal(cache["tile_ids"], cyprus.tile_id.to_numpy())
    assert str(cache["revision"]) == MODEL_REVISION
    assert str(cache["preprocessing"]) == PREPROCESS_ID
    cyprus_vectors = cache["embeddings"].copy()
cyprus["score"] = dino_probe.predict_proba(cyprus_vectors)[:, 1]
display(cyprus[["tile_id", "kind", "LocQual", "gsd_m", "clear_land_fraction", "score"]].sort_values("score", ascending=False))
ranked_cyprus = cyprus.sort_values("score", ascending=False)
contact_sheet(ranked_cyprus, ranked_cyprus.score.to_numpy(), n=12,
              title="Cyprus: transferred Enki scores; no archaeological visibility labels", show_labels=False)
display(json.loads((cyprus_dir / "provenance.json").read_text()))
cyprus.to_csv(ROOT / "cyprus_review.csv", index=False)


**Your transfer review:**
1. Compare a high-scoring background and a low-scoring catalogued location.
2. Identify at least three plausible causes of a source-to-Cyprus performance change.
3. Explain why treating all background chips as negatives would invalidate an accuracy claim.
4. Explain whether the scores support morphology detection or landscape-context ranking.
5. Specify the imagery, labels and independent geographic split needed for a real transfer study.

In [ ]:
transfer_review = {
    "high_scoring_background": "",
    "low_scoring_catalogued_location": "",
    "domain_shift_causes": [],
    "why_no_accuracy_score": "",
    "next_study": "",
}


### 10. Freeze your model and submit
Freeze the selected representation and classifier before touching the test set. Export one
score per test tile and your scientific recommendation. The test labels are recoverable from
the public archive, so this is an educational holdout rather than a secure competition.

In [ ]:
frozen_configuration = {
    "selected_model": best_key, "seed": SEED, "review_budget": REVIEW_BUDGET,
    "training_tiles": len(train), "test_tiles": len(test),
    "dataset": "Enki Normalized_Tell_combined_350.zip, curated 600-image workshop sample",
    "sample_manifest_sha256": file_digest(assets / "sample_manifest.json"),
    "encoder": MODEL_ID, "encoder_revision": MODEL_REVISION,
    "preprocessing": PREPROCESS_ID, "geographic_independence_verified": False,
}
test_scores = best_model.predict_proba(feature_sets[best_representation]["test"])[:, 1]
submission = write_submission(test, test_scores, ROOT / "submission.csv")
(ROOT / "configuration.json").write_text(json.dumps(frozen_configuration, indent=2))
display(submission.head())
print("Model frozen. Predictions saved to", ROOT / "submission.csv")


In [ ]:
recommendation = {
    "use_case": "",
    "validation_evidence": "",
    "main_limitations": [],
    "what_we_would_do_next": "",
}
# Replace the empty responses before sharing your submission.


In [ ]:
report = {"error_analysis": error_notes, "transfer_review": transfer_review,
          "recommendation": recommendation}
(ROOT / "team_report.json").write_text(json.dumps(report, indent=2))
output_zip = ROOT / "team_submission.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as out:
    for name in ("submission.csv", "configuration.json", "validation_results.csv",
                 "learning_curve.csv", "cyprus_review.csv", "team_report.json"):
        out.write(ROOT / name, name)
print("Download", output_zip, "from Colab's Files panel; save your completed notebook too.")


### 11. Final reveal — only after submission
Ask the facilitator when to enable this cell. Do not use the result to tune or replace your model.
Recall is measured on positive **tiles**, not independently verified archaeological sites.
For a random ranking, expected recall@k is k/N; expected precision is the positive fraction.
Neither this prevalence nor this small sample represents the operational archaeological base rate.

In [ ]:
REVEAL_TEST = False
if REVEAL_TEST:
    final_metrics = score_submission(submission, test, ranking_metrics, REVIEW_BUDGET)
    display(pd.Series(final_metrics, name="final_holdout"))
    print("Random ranking expected recall:", min(REVIEW_BUDGET, len(test)) / len(test))
    print("Random ranking expected precision:", test.label.mean())
    # A single diagnostic view after the model was frozen.
    order = ranking_indices(test_scores)
    contact_sheet(test.iloc[order], test_scores[order], n=12, title="Final top-ranked holdout tiles")
else:
    print("Test reveal disabled. Submit before asking the facilitator to reveal.")


### Sources, provenance and attribution
- Crawford & Vella (2022), [Cyprus Dataset: Settlements from 11000 BCE to 1878 CE](https://doi.org/10.5334/joad.96).
  [Source repository](https://github.com/K-A-Crawford/Cyprus_Settlements), pinned revision
  `70a5c3d399a2451e1d8dd5eb947eb80055276b44`. Dataset: CC BY 4.0.
- Rosati (2025), [Enki: AI for Archaeology — datasets](https://zenodo.org/records/14950565).
  We sample `Normalized_Tell_combined_350.zip` (published whole-archive MD5
  `1d4ea398ea5669f9bca4539416847206`). Partial downloads verify member CRCs and pixel hashes,
  not the full archive MD5. The embedded manifest records every selected member and byte range.
  The record declares CC BY-SA 4.0 and separately describes Google/Bing imagery provenance
  and provider conditions. Retain those notices when reusing its imagery or derived materials.
- Oquab et al., [DINOv2](https://github.com/facebookresearch/dinov2) and
  [model card](https://huggingface.co/facebook/dinov2-small). Model revision and preprocessing
  are recorded in the cache. The workshop adds no encoder fine-tuning.
- Cyprus chips: contains modified Copernicus Sentinel data (2024), accessed through
  [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a).
  Scene `S2A_MSIL2A_20240805T082601_R021_T36SVD_20240805T144559`.
  No commercial imagery account or Earth Engine authentication is needed for this sample.
- Follow-on EO model: [RemoteCLIP](https://github.com/ChenDelong1999/RemoteCLIP).
  It is not used to generate this notebook's reported results.

**Troubleshooting:** if Zenodo is temporarily unavailable or refuses byte ranges, use the
facilitator's `enki_workshop_sample.zip` in `archaeology_workshop/`; the member checks still
apply. Do not put the original full archive at that path. A restarted runtime loses local outputs
unless you download them. If a helper
name is undefined, rerun setup and the infrastructure cells in order. For the default path,
leave GPU recomputation and live Cyprus refresh disabled.